##  Pipeline Bronze - Ingestão de Dados

###  Visão Geral
Este notebook implementa a ingestão de dados na camada **Bronze** da arquitetura medalhão, utilizando arquivos armazenados no Data Lake.

---

###  Pré-requisitos (ordem de criação)
Antes de executar este pipeline, é necessário garantir a seguinte sequência:

1. **Catálogo criado (Unity Catalog)**  
2. **External Location configurada** (permissão de acesso ao Data Lake)  
3. **Volume criado** (apontando para o diretório de dados)

---

###  Arquitetura
- **Raw** → Arquivos brutos no Data Lake  
- **Bronze** → Dados ingeridos sem tratamento  
- **Silver** → Dados tratados  
- **Gold** → Dados agregados  

---

###  Objetivo
- Ingerir dados do Data Lake  
- Manter os dados no formato original  
- Criar base para transformações futuras  

---

###  Origem dos Dados
`abfss://bikesales@stgdatalakebr01.dfs.core.windows.net/raw`

---

###  Fluxo
1. Leitura dos arquivos no Data Lake  
2. Ingestão para Delta Lake (Bronze)  
3. Disponibilização para próximas camadas  

In [0]:
display(dbutils.fs.ls("abfss://bikesales@stgdatalakebr01.dfs.core.windows.net/"))

In [0]:
%sql
-- Garante que o schema existe (catalogo.schema) 
CREATE SCHEMA IF NOT EXISTS bikesales.raw;
CREATE SCHEMA IF NOT EXISTS bikesales.bronze;
CREATE SCHEMA IF NOT EXISTS bikesales.silver;
CREATE SCHEMA IF NOT EXISTS bikesales.gold;

In [0]:
%sql
-- RAW (arquivos brutos)
CREATE EXTERNAL VOLUME IF NOT EXISTS bikesales.raw.raw_files
LOCATION 'abfss://bikesales@stgdatalakebr01.dfs.core.windows.net/raw';

-- (opcional) bronze como staging de arquivos
CREATE EXTERNAL VOLUME IF NOT EXISTS bikesales.bronze.bronze_files
LOCATION 'abfss://bikesales@stgdatalakebr01.dfs.core.windows.net/bronze';

CREATE EXTERNAL VOLUME IF NOT EXISTS bikesales.silver.silver_files
LOCATION 'abfss://bikesales@stgdatalakebr01.dfs.core.windows.net/silver';

CREATE EXTERNAL VOLUME IF NOT EXISTS bikesales.gold.gold_files
LOCATION 'abfss://bikesales@stgdatalakebr01.dfs.core.windows.net/gold';

In [0]:
# Definir pastas do projeto em variáveis para facilitar
raw_path      = 'abfss://bikesales@stgdatalakebr01.dfs.core.windows.net/raw/'
bronze_path   = 'abfss://bikesales@stgdatalakebr01.dfs.core.windows.net/bronze/'
silver_path   = 'abfss://bikesales@stgdatalakebr01.dfs.core.windows.net/silver/'
gold_path     = 'abfss://bikesales@stgdatalakebr01.dfs.core.windows.net/gold/'


In [0]:
# Lista os arquivos na pasta raw
display(dbutils.fs.ls(raw_path))

In [0]:
from pyspark.sql.functions import current_date

# Lista com o nome exato dos arquivos
tabelas = [
    "brands", "categories", "customers", "order_items", 
    "orders", "products", "staffs", "stocks", "stores"
]

for tabela in tabelas:
    caminho_origem = f"{raw_path.strip('/')}/{tabela}.csv"
    caminho_destino = f"{bronze_path.strip('/')}/{tabela}"
    
    print(f"Processando: {tabela}.csv -> Bronze")
    
    try:
      
        df_temp = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .load(caminho_origem)
        
        # Adiciona a coluna de data corrente
        df_final = df_temp.withColumn("data_carga_bronze", current_date())
        
        # Gravação em formato Delta na camada Bronze
        df_final.write \
            .mode('overwrite') \
            .format('delta') \
            .option("mergeSchema", "true") \
            .save(caminho_destino)
            
        print(f" Sucesso: {tabela} carregada na Bronze.")
        
    except Exception as e:
        print(f" Erro ao processar {tabela}: {str(e)}")

print("\n Processo finalizado")

In [0]:
# Lendo um arquivo via caminho do Volume
df = spark.read.format("delta").load("/Volumes/bikesales/bronze/bronze_files/brands")
display(df)

In [0]:
# Listar os arquivos na pasta bronze
display(dbutils.fs.ls(bronze_path))

In [0]:
# 1. Definir o caminho base da bronze
path_bronze_base = "abfss://bikesales@stgdatalakebr01.dfs.core.windows.net/bronze/"

# 2. Listar todas as pastas dentro de bronze
pastas = dbutils.fs.ls(path_bronze_base)

for p in pastas:
    nome_tabela = p.name.strip('/')
    caminho_completo = p.path
    
    print(f"Registrando tabela: {nome_tabela}...")
    
    try:
        # Lendo os dados Delta da pasta
        df_temp = spark.read.format("delta").load(caminho_completo)
        
        # Criando a tabela no catálogo bikesales no schema bikes

        df_temp.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(f"bikesales.bronze.{nome_tabela}")
            
        print(f" Tabela {nome_tabela} criada com sucesso!")
        
    except Exception as e:
        print(f" Erro ao processar {nome_tabela}: {e}")

print("\n--- Processo de registro finalizado ---")